# 02. geolocation 대표좌표와 거리 계산 (Day 2)
**목적**: 우편번호별 대표좌표를 만들고, 고객·셀러 사이의 거리(distance_km)를 계산한다. 모집단 기준 테이블(base)도 여기서 만든다.

In [1]:
# 참고: 아래 셀에서 만드는 테이블(base, geo_clean, geo_rep, zip_status, item_lvl)은 재실행하면 목록에 함께 보임
import duckdb

con = duckdb.connect("data/interim/olist.duckdb")
con.execute("SHOW TABLES").df().name.tolist()

['base',
 'cat_tr',
 'chk',
 'customers',
 'geo_clean',
 'geo_rep',
 'geolocation',
 'item_lvl',
 'order_items',
 'order_payments',
 'order_reviews',
 'orders',
 'products',
 'rev',
 'sellers',
 'zip_status']

In [2]:
con.execute("""
CREATE OR REPLACE TABLE base AS
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp, order_approved_at,
       order_delivered_customer_date, order_estimated_delivery_date,
       CASE WHEN CAST(order_delivered_customer_date AS DATE)
                 > CAST(order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END AS late
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp >= '2017-01-01'
  AND order_purchase_timestamp <  '2018-09-01'
""")

con.execute("""
SELECT COUNT(*) AS n_orders, COUNT(DISTINCT order_id) AS n_unique,
       SUM(late) AS n_late, ROUND(100.0 * AVG(late), 2) AS late_pct
FROM base
""").df()

,n_orders,n_unique,n_late,late_pct
0,96204,96204,6532.0,6.79


In [3]:
display(con.execute("DESCRIBE geolocation").df()[["column_name", "column_type"]])

con.execute("""
SELECT COUNT(*) AS n_rows,
       MIN(geolocation_lat) AS min_lat, MAX(geolocation_lat) AS max_lat,
       MIN(geolocation_lng) AS min_lng, MAX(geolocation_lng) AS max_lng
FROM geolocation
""").df()

,column_name,column_type
0,geolocation_zip_code_prefix,INTEGER
1,geolocation_lat,DOUBLE
2,geolocation_lng,DOUBLE
3,geolocation_city,VARCHAR
4,geolocation_state,VARCHAR


,n_rows,min_lat,max_lat,min_lng,max_lng
0,1000163,-36.605374,45.065933,-101.466766,121.105394


In [4]:
con.execute("""
SELECT COUNT(*) AS n_zip,
       MIN(n) AS min_rows, MEDIAN(n) AS median_rows,
       ROUND(AVG(n), 1) AS mean_rows, MAX(n) AS max_rows,
       SUM(CASE WHEN n = 1 THEN 1 ELSE 0 END) AS zips_with_1_row
FROM (SELECT geolocation_zip_code_prefix, COUNT(*) AS n
      FROM geolocation GROUP BY geolocation_zip_code_prefix)
""").df()

,n_zip,min_rows,median_rows,mean_rows,max_rows,zips_with_1_row
0,19015,1,29.0,52.6,1146,1043.0


In [5]:
con.execute("""
SELECT geolocation_state AS state, COUNT(*) AS n_rows,
       COUNT(DISTINCT geolocation_zip_code_prefix) AS n_zips,
       ROUND(MIN(geolocation_lat), 1) AS min_lat, ROUND(MAX(geolocation_lat), 1) AS max_lat,
       ROUND(MIN(geolocation_lng), 1) AS min_lng, ROUND(MAX(geolocation_lng), 1) AS max_lng
FROM geolocation
WHERE NOT (geolocation_lat BETWEEN -34 AND 6 AND geolocation_lng BETWEEN -74 AND -28)
GROUP BY geolocation_state ORDER BY n_rows DESC
""").df()

,state,n_rows,n_zips,min_lat,max_lat,min_lng,max_lng
0,PA,7,3,38.7,42.4,-9.1,-6.9
1,RJ,5,4,-34.6,43.7,-58.7,13.8
2,BA,4,3,-34.6,39.0,-58.9,-4.9
3,RS,4,2,-36.6,14.6,-64.3,121.1
4,PR,3,2,39.1,42.2,-9.4,-8.7
5,MG,2,1,26.0,26.0,-98.1,-98.1
6,ES,2,1,21.7,29.4,-101.5,-98.5
7,PB,1,1,41.4,41.4,-8.7,-8.7
8,SP,1,1,28.0,28.0,-15.5,-15.5
9,MT,1,1,38.8,38.8,-9.4,-9.4


In [6]:
con.execute("""
SELECT geolocation_state AS state, geolocation_city AS city, COUNT(*) AS n_rows,
       ROUND(AVG(geolocation_lat), 2) AS avg_lat, ROUND(AVG(geolocation_lng), 2) AS avg_lng
FROM geolocation
WHERE geolocation_lat BETWEEN -34 AND 6 AND geolocation_lng > -34 AND geolocation_lng <= -28
GROUP BY geolocation_state, geolocation_city
""").df()

,state,city,n_rows,avg_lat,avg_lng
0,PE,fernando de noronha,11,-3.85,-32.41


## 범위 밖 좌표 제거 (D-09)
브라질 범위(위도 -34~6, 경도 -74~-28) 밖 좌표를 제거한다. 경도 -28은 페르난두 지 노로냐 섬(경도 약 -32.4)처럼 브라질령 섬을 포함하기 위한 값이다.

In [7]:
# [셀 7] geo_clean 생성 → 기대: 1,000,163 / 1,000,132 / 19,011
con.execute("""
CREATE OR REPLACE TABLE geo_clean AS
SELECT * FROM geolocation
WHERE geolocation_lat BETWEEN -34 AND 6
  AND geolocation_lng BETWEEN -74 AND -28
""")

con.execute("""
SELECT (SELECT COUNT(*) FROM geolocation) AS n_before,
       COUNT(*) AS n_after,
       COUNT(DISTINCT geolocation_zip_code_prefix) AS n_zip_after
FROM geo_clean
""").df()

,n_before,n_after,n_zip_after
0,1000163,1000132,19011


## 대표좌표: 평균 vs 중앙값 근거 (D-10)
박스 필터를 통과해도 브라질 안에서 엉뚱한 곳에 찍힌 좌표가 남는다. 평균은 그런 좌표에 끌려가고 중앙값은 잘 안 움직이므로, 대표좌표는 중앙값으로 정한다. 아래는 그 근거 숫자다.

In [8]:
# [셀 8] 평균 vs 중앙값 근거
# (a) 우편번호별 평균 대표점 vs 중앙값 대표점의 거리 차이 → 기대: 19,011 / 223 / 96 / 1,120
display(con.execute("""
WITH c AS (
  SELECT geolocation_zip_code_prefix AS zip,
         AVG(geolocation_lat) AS mean_lat, AVG(geolocation_lng) AS mean_lng,
         MEDIAN(geolocation_lat) AS med_lat, MEDIAN(geolocation_lng) AS med_lng
  FROM geo_clean GROUP BY geolocation_zip_code_prefix),
d AS (
  SELECT zip, 2 * 6371.0088 * asin(sqrt(
           pow(sin(radians(med_lat - mean_lat) / 2), 2) +
           cos(radians(mean_lat)) * cos(radians(med_lat)) *
           pow(sin(radians(med_lng - mean_lng) / 2), 2))) AS km
  FROM c)
SELECT COUNT(*) AS n_zip,
       SUM(CASE WHEN km > 10 THEN 1 ELSE 0 END) AS gt10km,
       SUM(CASE WHEN km > 50 THEN 1 ELSE 0 END) AS gt50km,
       ROUND(MAX(km), 0) AS max_km
FROM d
""").df())

# (b) 평균 대표점에서 가장 먼 좌표까지의 거리가 200km를 넘는 우편번호 → 기대: 19,011 / 207 / 2,994
display(con.execute("""
WITH c AS (
  SELECT geolocation_zip_code_prefix AS zip,
         AVG(geolocation_lat) AS mean_lat, AVG(geolocation_lng) AS mean_lng
  FROM geo_clean GROUP BY geolocation_zip_code_prefix),
d AS (
  SELECT g.geolocation_zip_code_prefix AS zip,
         2 * 6371.0088 * asin(sqrt(
           pow(sin(radians(g.geolocation_lat - c.mean_lat) / 2), 2) +
           cos(radians(c.mean_lat)) * cos(radians(g.geolocation_lat)) *
           pow(sin(radians(g.geolocation_lng - c.mean_lng) / 2), 2))) AS km
  FROM geo_clean g JOIN c ON c.zip = g.geolocation_zip_code_prefix),
m AS (SELECT zip, MAX(km) AS max_km FROM d GROUP BY zip)
SELECT COUNT(*) AS n_zip,
       SUM(CASE WHEN max_km > 200 THEN 1 ELSE 0 END) AS zips_with_point_gt200km,
       ROUND(MAX(max_km), 0) AS max_km
FROM m
""").df())

,n_zip,gt10km,gt50km,max_km
0,19011,223.0,96.0,1120.0


,n_zip,zips_with_point_gt200km,max_km
0,19011,207.0,2994.0


## 대표좌표 테이블 geo_rep
우편번호당 1행(중앙값 위·경도 + 관측치 수 n). 조인 전에 이 표부터 만들어야 행이 폭발하지 않는다.

In [9]:
# [셀 9] geo_rep 생성 → 기대: 19,011 / 19,011 / 1,042 (앞 두 숫자가 같아야 우편번호당 1행)
con.execute("""
CREATE OR REPLACE TABLE geo_rep AS
SELECT geolocation_zip_code_prefix AS zip,
       MEDIAN(geolocation_lat) AS lat,
       MEDIAN(geolocation_lng) AS lng,
       COUNT(*) AS n
FROM geo_clean
GROUP BY geolocation_zip_code_prefix
""")

con.execute("""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT zip) AS n_zip,
       SUM(CASE WHEN n = 1 THEN 1 ELSE 0 END) AS zips_with_1_row
FROM geo_rep
""").df()

,n_rows,n_zip,zips_with_1_row
0,19011,19011,1042.0


## 매칭률 4방향 분해 (D-04)
고객·셀러 우편번호가 geo_rep에 없는 경우를 우편번호·행·base 주문 단위로 센다.
- **A**: geolocation 원본에 우편번호 자체가 없음
- **B**: 좌표는 있었는데 범위 필터로 전부 제거됨

미매칭은 NULL로 두고, 대체(동일 주 평균 + 결측 더미)는 학습 구간이 정해진 뒤에 한다.

In [10]:
# [셀 10] 매칭률 → 기대: customer zip 14,994/158/157/1, row 99,441/279/278/1, base_order 96,204/265/264/1
#                       seller zip 2,246/7/7/0, row 3,095/7/7/0, base_order 96,204/217/217/0
#                       거리 계산 불가 base 주문 96,204 / 481 / 0.5
con.execute("""
CREATE OR REPLACE TABLE zip_status AS
SELECT z.zip,
       CASE WHEN r.zip IS NOT NULL THEN 'ok'
            WHEN g.zip IS NOT NULL THEN 'B_filtered'
            ELSE 'A_not_in_geolocation' END AS status
FROM (SELECT customer_zip_code_prefix AS zip FROM customers
      UNION
      SELECT seller_zip_code_prefix FROM sellers) z
LEFT JOIN geo_rep r ON r.zip = z.zip
LEFT JOIN (SELECT DISTINCT geolocation_zip_code_prefix AS zip FROM geolocation) g ON g.zip = z.zip
""")

display(con.execute("""
SELECT 'customer' AS side, 'zip' AS level, COUNT(*) AS total,
       CAST(SUM(CAST(status <> 'ok' AS INTEGER)) AS INTEGER) AS unmatched,
       CAST(SUM(CAST(status = 'A_not_in_geolocation' AS INTEGER)) AS INTEGER) AS A,
       CAST(SUM(CAST(status = 'B_filtered' AS INTEGER)) AS INTEGER) AS B
FROM (SELECT DISTINCT customer_zip_code_prefix AS zip FROM customers) t JOIN zip_status USING (zip)
UNION ALL
SELECT 'customer', 'row', COUNT(*),
       CAST(SUM(CAST(status <> 'ok' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'A_not_in_geolocation' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'B_filtered' AS INTEGER)) AS INTEGER)
FROM customers c JOIN zip_status s ON s.zip = c.customer_zip_code_prefix
UNION ALL
SELECT 'customer', 'base_order', COUNT(*),
       CAST(SUM(CAST(status <> 'ok' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'A_not_in_geolocation' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'B_filtered' AS INTEGER)) AS INTEGER)
FROM base b JOIN customers c USING (customer_id) JOIN zip_status s ON s.zip = c.customer_zip_code_prefix
UNION ALL
SELECT 'seller', 'zip', COUNT(*),
       CAST(SUM(CAST(status <> 'ok' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'A_not_in_geolocation' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'B_filtered' AS INTEGER)) AS INTEGER)
FROM (SELECT DISTINCT seller_zip_code_prefix AS zip FROM sellers) t JOIN zip_status USING (zip)
UNION ALL
SELECT 'seller', 'row', COUNT(*),
       CAST(SUM(CAST(status <> 'ok' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'A_not_in_geolocation' AS INTEGER)) AS INTEGER),
       CAST(SUM(CAST(status = 'B_filtered' AS INTEGER)) AS INTEGER)
FROM sellers s JOIN zip_status z ON z.zip = s.seller_zip_code_prefix
UNION ALL
SELECT 'seller', 'base_order', COUNT(*), CAST(SUM(any_bad) AS INTEGER),
       CAST(SUM(any_a) AS INTEGER), CAST(SUM(any_b) AS INTEGER)
FROM (SELECT b.order_id,
             MAX(CAST(z.status <> 'ok' AS INTEGER)) AS any_bad,
             MAX(CAST(z.status = 'A_not_in_geolocation' AS INTEGER)) AS any_a,
             MAX(CAST(z.status = 'B_filtered' AS INTEGER)) AS any_b
      FROM base b
      JOIN order_items i ON i.order_id = b.order_id
      JOIN sellers s ON s.seller_id = i.seller_id
      JOIN zip_status z ON z.zip = s.seller_zip_code_prefix
      GROUP BY b.order_id)
""").df())

# 고객 또는 (어떤 아이템의) 셀러가 미매칭이라 거리를 못 구하는 base 주문
display(con.execute("""
SELECT COUNT(*) AS n_base_orders, SUM(bad) AS n_no_distance, ROUND(100.0 * AVG(bad), 2) AS pct
FROM (SELECT b.order_id,
             MAX(CASE WHEN zc.status <> 'ok' OR zs.status <> 'ok' THEN 1 ELSE 0 END) AS bad
      FROM base b
      JOIN customers c USING (customer_id)
      JOIN order_items i ON i.order_id = b.order_id
      JOIN sellers s ON s.seller_id = i.seller_id
      JOIN zip_status zc ON zc.zip = c.customer_zip_code_prefix
      JOIN zip_status zs ON zs.zip = s.seller_zip_code_prefix
      GROUP BY b.order_id)
""").df())

,side,level,total,unmatched,A,B
0,customer,zip,14994,158,157,1
1,customer,row,99441,279,278,1
2,customer,base_order,96204,265,264,1
3,seller,zip,2246,7,7,0
4,seller,row,3095,7,7,0
5,seller,base_order,96204,217,217,0


,n_base_orders,n_no_distance,pct
0,96204,481.0,0.5


## item_lvl: 아이템 단위 distance_km (하버사인)
고객·셀러 대표좌표로 두 점 사이 직선거리를 구한다. `LEFT JOIN`이라 좌표가 없으면 행은 남고 `distance_km`만 NULL이 된다. 조인 전후 행 수(112,650)가 같아야 한다.

`item_lvl`은 기간·배송 여부와 무관하게 order_items 전체(112,650행) 기준이다. 분석 모집단(base)은 Day 3에서 조인할 때 한정한다.

In [11]:
# [셀 11] item_lvl 생성 → 기대: 112,650 / 112,650 / 112,095 / 555 (앞 두 숫자가 같아야 함)
con.execute("""
CREATE OR REPLACE TABLE item_lvl AS
SELECT i.order_id, i.order_item_id, i.product_id, i.seller_id, i.price, i.freight_value,
       c.customer_zip_code_prefix AS customer_zip, s.seller_zip_code_prefix AS seller_zip,
       rc.n AS c_n, rs.n AS s_n,
       2 * 6371.0088 * asin(sqrt(
           pow(sin(radians(rs.lat - rc.lat) / 2), 2) +
           cos(radians(rc.lat)) * cos(radians(rs.lat)) *
           pow(sin(radians(rs.lng - rc.lng) / 2), 2))) AS distance_km
FROM order_items i
JOIN orders    o ON o.order_id    = i.order_id
JOIN customers c ON c.customer_id = o.customer_id
JOIN sellers   s ON s.seller_id   = i.seller_id
LEFT JOIN geo_rep rc ON rc.zip = c.customer_zip_code_prefix
LEFT JOIN geo_rep rs ON rs.zip = s.seller_zip_code_prefix
""")

con.execute("""
SELECT (SELECT COUNT(*) FROM order_items) AS n_before, COUNT(*) AS n_after,
       COUNT(distance_km) AS n_with_distance, COUNT(*) - COUNT(distance_km) AS n_null
FROM item_lvl
""").df()

,n_before,n_after,n_with_distance,n_null
0,112650,112650,112095,555


## 거리 분포 검증
이상 신호는 4,500km 초과(브라질 본토 안 최장 직선거리가 4,300km대). 정상 값도 3,500km대까지 나오므로 "수천 km"를 기준으로 삼지 않는다.

In [12]:
# [셀 12] 거리 분포·공식 검증·n=1 우편번호·셀러 지역 집중도
# (a) 기대: min 0 / p25 183 / median 431.7 / p75 792 / p90 1,451 / max 3,579 / 4,500km 초과 0건
display(con.execute("""
SELECT ROUND(MIN(distance_km), 1) AS min_km,
       ROUND(QUANTILE_CONT(distance_km, 0.25), 0) AS p25,
       ROUND(MEDIAN(distance_km), 1) AS median_km,
       ROUND(QUANTILE_CONT(distance_km, 0.75), 0) AS p75,
       ROUND(QUANTILE_CONT(distance_km, 0.90), 0) AS p90,
       ROUND(MAX(distance_km), 0) AS max_km,
       SUM(CASE WHEN distance_km > 4500 THEN 1 ELSE 0 END) AS n_gt_4500km
FROM item_lvl
""").df())

# (b) 공식 검증: 상파울루 ↔ 리우 → 기대: 360.7
display(con.execute("""
SELECT ROUND(2 * 6371.0088 * asin(sqrt(
         pow(sin(radians(-22.9068 - (-23.5505)) / 2), 2) +
         cos(radians(-23.5505)) * cos(radians(-22.9068)) *
         pow(sin(radians(-43.1729 - (-46.6333)) / 2), 2))), 1) AS sao_paulo_to_rio_km
""").df())

# (c) 관측치가 1개뿐인 우편번호에 기대는 아이템 → 기대: 539 / 0.48
display(con.execute("""
SELECT SUM(CASE WHEN c_n = 1 OR s_n = 1 THEN 1 ELSE 0 END) AS items_with_n1_zip,
       ROUND(100.0 * AVG(CASE WHEN c_n = 1 OR s_n = 1 THEN 1 ELSE 0 END), 2) AS pct
FROM item_lvl
""").df())

# (d) 셀러 지역 집중도 → 기대: 셀러 수 기준 SP 1,849(59.7%) / 아이템 기준 SP 80,342(71.3%)
display(con.execute("""
SELECT seller_state, COUNT(*) AS n_sellers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_sellers
FROM sellers GROUP BY seller_state ORDER BY n_sellers DESC LIMIT 5
""").df())
display(con.execute("""
SELECT s.seller_state, COUNT(*) AS n_items,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_items
FROM order_items i JOIN sellers s USING (seller_id)
GROUP BY s.seller_state ORDER BY n_items DESC LIMIT 5
""").df())

,min_km,p25,median_km,p75,p90,max_km,n_gt_4500km
0,0.0,183.0,431.7,792.0,1451.0,3579.0,0.0


,sao_paulo_to_rio_km
0,360.7


,items_with_n1_zip,pct
0,539.0,0.48


,seller_state,n_sellers,pct_sellers
0,SP,1849,59.7
1,PR,349,11.3
2,MG,244,7.9
3,SC,190,6.1
4,RJ,171,5.5


,seller_state,n_items,pct_items
0,SP,80342,71.3
1,MG,8827,7.8
2,PR,8671,7.7
3,RJ,4818,4.3
4,SC,4075,3.6


## 보충: 미매칭 셀러의 주문 수 (D-04)
미매칭 셀러 7곳이 어느 정도 규모의 셀러인지 본다. 평균만 비교하면 분포가 치우쳐 있어 오해할 수 있으므로, 셀러별 주문 수와 전체 분포(중앙값, 순위)를 함께 확인한다.

In [13]:
# [보충] 미매칭 셀러 7곳의 base 주문 수 (D-04)
# (a) 기대: 127 / 45 / 34 / 6 / 2 / 2 / 1 (합 217)
display(con.execute("""
SELECT s.seller_id, s.seller_zip_code_prefix AS zip, COUNT(DISTINCT b.order_id) AS n_orders
FROM sellers s
JOIN order_items i ON i.seller_id = s.seller_id
JOIN base b ON b.order_id = i.order_id
JOIN zip_status z ON z.zip = s.seller_zip_code_prefix
WHERE z.status <> 'ok'
GROUP BY s.seller_id, s.seller_zip_code_prefix
ORDER BY n_orders DESC
""").df())

# (b) 전체 셀러 주문 수 분포와 미매칭 최대 셀러(127건)의 순위
#     기대: 2,945 / 평균 33.12 / 중앙값 7 / 최대 1,819 / 127 / 159곳 / 5.4%
con.execute("""
WITH sc AS (
  SELECT i.seller_id, COUNT(DISTINCT b.order_id) AS n_orders
  FROM base b JOIN order_items i ON i.order_id = b.order_id
  GROUP BY i.seller_id),
um AS (
  SELECT sc.seller_id, sc.n_orders
  FROM sc JOIN sellers s USING (seller_id)
          JOIN zip_status z ON z.zip = s.seller_zip_code_prefix
  WHERE z.status <> 'ok')
SELECT (SELECT COUNT(*) FROM sc) AS n_sellers,
       ROUND((SELECT AVG(n_orders) FROM sc), 2) AS mean_orders,
       (SELECT MEDIAN(n_orders) FROM sc) AS median_orders,
       (SELECT MAX(n_orders) FROM sc) AS max_orders,
       (SELECT MAX(n_orders) FROM um) AS top_unmatched,
       (SELECT COUNT(*) FROM sc WHERE n_orders >= (SELECT MAX(n_orders) FROM um)) AS sellers_ge_top,
       ROUND(100.0 * (SELECT COUNT(*) FROM sc WHERE n_orders >= (SELECT MAX(n_orders) FROM um))
             / (SELECT COUNT(*) FROM sc), 1) AS pct_ge_top
""").df()

,seller_id,zip,n_orders
0,2e90cb1677d35cfe24eef47d441b7c87,2285,127
1,870d0118f7a9d85960f29ad89d5d989a,37708,45
2,42bde9fef835393bb8a8849cb6b7f245,71551,34
3,5962468f885ea01a1b6a97a218797b0a,82040,6
4,2aafae69bf4c41fbd94053d9413e87ee,91901,2
5,0b3f27369a4d8df98f7eb91077e438ac,7412,2
6,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,1


,n_sellers,mean_orders,median_orders,max_orders,top_unmatched,sellers_ge_top,pct_ge_top
0,2945,33.12,7.0,1819,127,159,5.4


## 결론
- 대표좌표는 우편번호별 **중앙값**으로 확정(D-10). 평균과 10km 넘게 다른 우편번호가 223개, 평균에서 200km 넘는 좌표가 섞인 우편번호가 207개(최대 2,994km)
- 미매칭은 거의 전부 geolocation에 우편번호 자체가 없는 경우(A). 고객 base 주문 265건(0.28%), 셀러 217건(0.23%, 셀러 7곳 중 3곳이 95%), 거리 계산 불가 주문 481건(0.5%). 범위 필터로 인한 미매칭(B)은 고객 1건뿐
- item_lvl 112,650행(조인 전후 동일), distance_km NULL 555건. 거리 중앙값 431.7km, 90% 1,451km, 최대 3,579km로 긴 꼬리. 셀러는 상파울루(SP)에 집중(셀러 59.7%, 아이템 71.3%)

In [14]:
con.close()